# Stage 9–10: Evaluation and Hyperparameter Tuning
Metrics: MAE, MSE, RMSE and R². Regression tasks do not use classification accuracy.

In [1]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run notebook 05 first in this same kernel to create these variables.


In [2]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])
param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}
search = RandomizedSearchCV(
    rf_pipe, param_distributions=param_dist, n_iter=8, cv=5,
    scoring="neg_root_mean_squared_error", random_state=42,
    n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)
print("Tuned Random Forest CV RMSE:", -search.best_score_)
print("Best parameters:", search.best_params_)

# Compare the tuned Random Forest with the best baseline model from notebook 05.
cv_results_df = pd.DataFrame(cv_results).sort_values("CV_RMSE")
best_baseline_name = cv_results_df.iloc[0]["Model"]
baseline_estimator = models[best_baseline_name]
baseline_model = Pipeline([("prep", preprocessor), ("model", baseline_estimator)])
baseline_model.fit(X_train, y_train)
baseline_cv_rmse = float(cv_results_df.iloc[0]["CV_RMSE"])
tuned_cv_rmse = float(-search.best_score_)

if tuned_cv_rmse <= baseline_cv_rmse:
    best_model = search.best_estimator_
    selected_model_name = "Tuned Random Forest"
else:
    best_model = baseline_model
    selected_model_name = best_baseline_name

print("Selected model:", selected_model_name)


NameError: name 'preprocessor' is not defined

In [ ]:
predictions = best_model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
metrics = {
    "MAE": float(mean_absolute_error(y_test, predictions)),
    "MSE": float(mse),
    "RMSE": float(np.sqrt(mse)),
    "R2": float(r2_score(y_test, predictions))
}
print("Held-out test metrics:", metrics)
metrics


In [ ]:
comparison = pd.DataFrame({"actual": y_test, "predicted": predictions})
comparison.head(10)

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, predictions, alpha=.5)
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("Actual vs Predicted")
plt.tight_layout()
plt.show()

## Interpret results
- Lower MAE/RMSE is better.
- R² closer to 1 is generally better; negative R² means the model performs worse than predicting the test-set mean.
- Explain errors in the target's unit.
- Do not report results until you have run the notebook on your real downloaded dataset.